In [12]:
import os
from dotenv import load_dotenv
import requests
load_dotenv()

True

# Defining research tools

### ArXiv search tool

In [ ]:
import xml.etree.ElementTree as ET

def arxiv_search_tool(query: str, max_results: int=5) -> list[dict]:
    # arXiv API search url with query and max_results parametrs (eg. query="big data", max_results=2)
    url = f"https://export.arxiv.org/api/query?search_query=all:{query}&start=0&max_results={max_results}"
    response = requests.get(url)
    
    # Since arXiv returns XML, convert XML response into python readable structure
    root = ET.fromstring(response.content)
    
    # XML namespace for arXiv Atom feed, needed to access tags like title, author etc.
    ns = {"atom": "http://www.w3.org/2005/Atom"}
    
    results = []
    
    # Loop through each paper result one by one
    for entry in root.findall("atom:entry", ns):
        # Extract paper tile
        title = entry.find("atom:title", ns).text.strip()
        
        # Extract all author names
        authors = [
            author.find("atom:name", ns).text
            for author in entry.findall("atom:author", ns)
        ]
        
        # Extract published date, [:10] keeps it int YYYY-MM-DD format
        published = entry.find("atom:published", ns).text[:10]
        
        # Extract paper abstract
        summary = entry.find("atom:summary", ns).text.strip()
        
        # Extract paper link
        article_url = entry.find("atom:id", ns).text
        
        results.append({
            "title":title,
            "authors":authors,
            "published":published,
            "summary":summary,
            "url":article_url
        })
        
    return results

In [3]:
arxiv_tool_def = {
    "type": "function",
    "function": {
        "name": "arxiv_search_tool",
        "description": "Searches for research papers on arXiv by query string.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search keywords for research papers."
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to return.",
                    "default": 5
                }
            },
            "required": ["query"]
        }
    }
}

### Tavily search tool

In [14]:
from tavily import TavilyClient

def tavily_search_tool(query: str, max_results: int = 5) -> list[dict]:
    # Create tavily client using api key
    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)
    
    # Perform Tavily web search
    response = client.search(
        query=query,
        max_results=max_results
    )
    
    results = []
    
    # Loop through each search result and store title, content and url
    for item in response["results"]:
        results.append({
            "title": item["title"],
            "content": item["content"],
            "url": item["url"]
        })
    
    return results

In [6]:
tavily_tool_def = {
    "type": "function",
    "function": {
        "name": "tavily_search_tool",
        "description": "Performs a general-purpose web search using the Tavily API.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search keywords for retrieving information from the web."
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to return.",
                    "default": 5
                },
                "include_images": {
                    "type": "boolean",
                    "description": "Whether to include image results.",
                    "default": False
                }
            },
            "required": ["query"]
        }
    }
}

### Wikipedia search tool

In [15]:
import wikipedia

def wikipedia_search_tool(query: str, sentences: int=5) -> list[dict]:
    # Searches wikipedia and takes first result title
    page_title = wikipedia.search(query)[0]
    
    # Opens the full wikipedia page
    page = wikipedia.page(page_title)
    
    # Gets summary using choosen number of sentences
    summary = wikipedia.summary(page_title, sentences=sentences)
    
    return [{
        "title": page.title,
        "summary": summary,
        "url": page.url
    }]
    

In [8]:
wikipedia_tool_def = {
    "type": "function",
    "function": {
        "name": "wikipedia_search_tool",
        "description": "Searches for a Wikipedia article summary by query string.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search keywords for the Wikipedia article."
                },
                "sentences": {
                    "type": "integer",
                    "description": "Number of sentences in the summary.",
                    "default": 5
                }
            },
            "required": ["query"]
        }
    }
}

In [16]:
tool_mapping = {
    "tavily_search_tool": tavily_search_tool,
    "arxiv_search_tool": arxiv_search_tool,
    "wikipedia_search_tool": wikipedia_search_tool
}

# Research step - finding references

In [17]:
from datetime import datetime
from aisuite import Client
client = Client()
openai_api_key = os.getenv("OPENAI_API_KEY")

def find_references(task:str, return_messages:bool=False):
    prompt = f"""
    You are a research function with access to:
    - arxiv_tool: academic papers
    - tavily_tool: general web search (return JSON when asked)
    - wikipedia_tool: encyclopedic summaries

    Task:
    {task}

    Today is {datetime.now().strftime('%Y-%m-%d')}.
    """.strip()
    
    messages = [
        {
            "role":"user",
            "content":prompt
        }
    ]
    
    tools = [
        arxiv_search_tool,
        tavily_search_tool,
        wikipedia_search_tool
    ]
    
    response = client.chat.completions.create(
        model="openai:gpt-4o",
        messages=messages,
        tools=tools,
        tool_choice="auto",
        max_turns=5
    )
    
    content = response.choices[0].message.content
    
    return (content, messages) if return_messages else content

In [18]:
research_result = find_references("Find 2 recent papers about recent developments in black hole science")
print(research_result)

Here are two papers on recent developments in black hole science:

1. **Title:** Accretion onto Supermassive Black Holes in Quasars: Learning from Optical/UV Observations
   - **Authors:** Paola Marziani, Deborah Dultzin-Hacyan, Jack W. Sulentic
   - **Publication Date:** June 28, 2006
   - **Summary:** This paper discusses the complex accretion processes in quasars and active galactic nuclei, focusing on the correlations between observed spectral properties and physical parameters like black hole mass, Eddington ratio, and spin. It highlights recent observational trends emerging from large spectral datasets and reviews the uncertainties involved in current estimates.
   - **URL:** [Read more](http://arxiv.org/abs/astro-ph/0606678v1)

2. **Title:** Disturbing the Black Hole
   - **Authors:** Jacob D. Bekenstein
   - **Publication Date:** May 13, 1998
   - **Summary:** The paper supports the conjecture that the horizon area of a near-equilibrium black hole is an adiabatic invariant. It 

# Evaluation Step - Preferred Domains

In [19]:
# list of preferred domains for Tavily results
TOP_DOMAINS = {
    # General reference / institutions / publishers
    "wikipedia.org", "nature.com", "science.org", "sciencemag.org", "cell.com",
    "mit.edu", "stanford.edu", "harvard.edu", "nasa.gov", "noaa.gov", "europa.eu",

    # CS/AI venues & indexes
    "arxiv.org", "acm.org", "ieee.org", "neurips.cc", "icml.cc", "openreview.net",

    # Other reputable outlets
    "elifesciences.org", "pnas.org", "jmlr.org", "springer.com", "sciencedirect.com",

    # Extra domains (case-specific additions)
    "pbs.org", "nova.edu", "nvcc.edu", "cccco.edu",

    # Well known programming sites
    "codecademy.com", "datacamp.com"
}

In [22]:
import re

def evaluate_tavily_results(TOP_DOMAINS, raw:str, min_ratio=0.4):
    # Extract urls from text
    url_pattern = re.compile(r'https?://[^\s\]\)>\}]+', flags=re.IGNORECASE)
    urls = url_pattern.findall(raw)
    
    # Count preferred vs total urls
    total  = len(urls)
    preferred_count = 0
    details = []
    
    for url in urls:
        domain = url.split("/")[2]
        preferred = any(td in domain for td in TOP_DOMAINS)
        if preferred:
            preferred_count += 1
        details.append(f"- {url} → {'✅ PREFERRED' if preferred else '❌ NOT PREFERRED'}")
    
    ratio = preferred_count / total if total > 0 else 0.0
    flag = ratio >= min_ratio
    
    # Create report
    report = f"""
    Evaluation — Tavily Preferred Domains for
    - Total results: {total}
    - Preferred results: {preferred_count}
    - Ratio: {ratio:.2%}
    - Threshold: {min_ratio:.0%}
    - Status: {"✅ PASS" if flag else "❌ FAIL"}
    
    **Details:**
    {chr(10).join(details)}
    """
    
    return flag, report
    

In [23]:
flag, report = evaluate_tavily_results(TOP_DOMAINS, research_result)
print(report)


    Evaluation — Tavily Preferred Domains for
    - Total results: 2
    - Preferred results: 2
    - Ratio: 100.00%
    - Threshold: 40%
    - Status: ✅ PASS

    **Details:**
    - http://arxiv.org/abs/astro-ph/0606678v1 → ✅ PREFERRED
- http://arxiv.org/abs/gr-qc/9805045v1 → ✅ PREFERRED
    


# Executing pipeline on another example

In [24]:
research_result = find_references("Find recent papers in diffusion models")
print(research_result)
flag, report = evaluate_tavily_results(TOP_DOMAINS, research_result)
print(report)

Here are some recent academic papers on diffusion models:

1. **Generalized Contrastive Divergence: Joint Training of Energy-Based Model and Diffusion Model through Inverse Reinforcement Learning**
   - **Authors**: Sangwoong Yoon, Dohyun Kwon, Himchan Hwang, Yung-Kyun Noh, Frank C. Park
   - **Published**: 2023-12-06
   - **Summary**: This paper introduces Generalized Contrastive Divergence (GCD), a new objective function for training an energy-based model (EBM) and a diffusion model simultaneously. The training is formulated as a minimax problem resembling inverse reinforcement learning. This method enables EBM training without MCMC while enhancing diffusion model sample quality. [Read more](http://arxiv.org/abs/2312.03397v1).

2. **Fixed Point Diffusion Models**
   - **Authors**: Xingjian Bai, Luke Melas-Kyriazi
   - **Published**: 2024-01-16
   - **Summary**: This research introduces the Fixed Point Diffusion Model (FPDM), integrating fixed point solving into diffusion models for i